## Part 1: Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [3]:
# Determine the number of unique values in each column
attrition_df.nunique()

,0
Age,43
Attrition,2
BusinessTravel,3
Department,3
DistanceFromHome,29
Education,5
EducationField,6
EnvironmentSatisfaction,4
HourlyRate,71
JobInvolvement,4


In [4]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']]
y_df.head()

,Attrition,Department
0,Yes,Sales
1,No,Research & Development
2,Yes,Research & Development
3,No,Research & Development
4,No,Research & Development


In [6]:
# Create a list of at least 10 column names to use as X data

X_df = attrition_df[['Age', 'OverTime', 'DistanceFromHome', 'Education', 'EducationField', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction']]

# Create X_df using your selected columns

X_df.head()

# Show the data types for X_df

X_df.dtypes

,0
Age,int64
OverTime,object
DistanceFromHome,int64
Education,int64
EducationField,object
EnvironmentSatisfaction,int64
HourlyRate,int64
JobInvolvement,int64
JobLevel,int64
JobSatisfaction,int64


In [7]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split


In [8]:
# Convert your X data to numeric data types however you see fit

X_df = pd.get_dummies(X_df)

X_df.head()

# Add new code cells as necessary


,Age,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,OverTime_No,OverTime_Yes,EducationField_Human Resources,EducationField_Life Sciences,EducationField_Marketing,EducationField_Medical,EducationField_Other,EducationField_Technical Degree
0,41,1,2,2,94,3,2,4,False,True,False,True,False,False,False,False
1,49,8,1,3,61,2,2,2,True,False,False,True,False,False,False,False
2,37,2,2,4,92,2,1,3,False,True,False,False,False,False,True,False
3,33,3,4,4,56,3,1,3,False,True,False,True,False,False,False,False
4,27,2,1,1,40,3,1,2,True,False,False,False,False,True,False,False


In [10]:
# Create a StandardScaler
from sklearn.preprocessing import StandardScaler
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, random_state=1)
StandardScaler = StandardScaler()
# Fit the StandardScaler to the training data
StandardScaler.fit(X_train)

# Scale the training and testing data
X_train_scaled = StandardScaler.transform(X_train)
X_test_scaled = StandardScaler.transform(X_test)

In [11]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder for the Department column
department_encoder  = OneHotEncoder(sparse_output=False)

# Fit the encoder to the training data
department_encoded = department_encoder.fit_transform(y_train['Department'].values.reshape(-1, 1))

# Create two new variables by applying the encoder
# to the training and testing data
y_train_dept_encoded = pd.DataFrame(department_encoded, columns=department_encoder.get_feature_names_out(['Department']))
y_test_dept_encoded = pd.DataFrame(department_encoder.transform(y_test['Department'].values.reshape(-1, 1)), columns=department_encoder.get_feature_names_out(['Department']))

In [12]:
# Create a OneHotEncoder for the Attrition column
attrition_encoder = OneHotEncoder(sparse_output=False)

# Fit the encoder to the training data
attrition_encoded = attrition_encoder.fit_transform(y_train['Attrition'].values.reshape(-1, 1))

# Create two new variables by applying the encoder
# to the training and testing data

y_train_attrition_encoded = pd.DataFrame(attrition_encoded, columns=attrition_encoder.get_feature_names_out(['Attrition']))
y_test_attrition_encoded = pd.DataFrame(attrition_encoder.transform(y_test['Attrition'].values.reshape(-1, 1)), columns=attrition_encoder.get_feature_names_out(['Attrition']))

## Part 2: Create, Compile, and Train the Model

In [13]:
# Find the number of columns in the X training data.
no_of_columns = X_train_scaled.shape[1]
no_of_columns

# Create the input layer
input_layer = layers.Input(shape=(no_of_columns,))

# Create at least two shared layers
shared_layer1 = layers.Dense(units=64, activation='relu')(input_layer)
shared_layer2 = layers.Dense(units=32, activation='relu')(shared_layer1)

In [14]:
# Create a branch for Department
# with a hidden layer and an output layer

# Create the hidden layer

department_hidden = layers.Dense(units=32, activation='relu')(shared_layer2)

# Create the output layer
department_output = layers.Dense(units=y_train_dept_encoded.shape[1], activation='softmax', name='department_output')(department_hidden)

In [15]:
# Create a branch for Attrition
# with a hidden layer and an output layer

# Create the hidden layer
attrition_hidden = layers.Dense(units=32, activation='relu')(shared_layer2)

# Create the output layer
attrition_output = layers.Dense(units=y_train_attrition_encoded.shape[1], activation='softmax', name='attrition_output')(attrition_hidden)

In [16]:
# Create the model
model = Model(inputs=input_layer, outputs=[department_output, attrition_output])

# Compile the model
model.compile(optimizer='adam',
              loss={'department_output':'categorical_crossentropy', 'attrition_output':'categorical_crossentropy'},
              metrics={'department_output':'accuracy', 'attrition_output':'accuracy'}
              )

# Summarize the model
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      1,088 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      1,056 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32)        │      1,056 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ department_output   │ (None, 3)         │         99 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attrition_output    │ (None, 2)         │         66 │ dense_3[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,445 (21.27 KB)

 Trainable params: 5,445 (21.27 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Train the model

model.fit(X_train_scaled, [y_train_dept_encoded, y_train_attrition_encoded], epochs=100, batch_size=32, validation_split=0.2)

Epoch 1/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - attrition_output_accuracy: 0.8509 - attrition_output_loss: 0.4716 - department_output_accuracy: 0.6142 - department_output_loss: 0.9292 - loss: 1.4012 - val_attrition_output_accuracy: 0.8281 - val_attrition_output_loss: 0.4846 - val_department_output_accuracy: 0.7285 - val_department_output_loss: 0.7537 - val_loss: 1.2413
Epoch 2/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - attrition_output_accuracy: 0.8408 - attrition_output_loss: 0.4221 - department_output_accuracy: 0.7744 - department_output_loss: 0.6726 - loss: 1.0948 - val_attrition_output_accuracy: 0.8190 - val_attrition_output_loss: 0.4566 - val_department_output_accuracy: 0.7330 - val_department_output_loss: 0.6793 - val_loss: 1.1389
Epoch 3/100
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - attrition_output_accuracy: 0.8741 - attrition_output_loss: 0.3364 - department_output_accuracy: 0.7532 - department_output_loss: 0.6234 - loss: 0.9597 - val_attrition_output_accuracy: 0.8281 -

In [19]:
# Evaluate the model with the testing data
test_results = model.evaluate(X_test_scaled, [y_test_dept_encoded, y_test_attrition_encoded])

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - attrition_output_accuracy: 0.8136 - attrition_output_loss: 1.3241 - department_output_accuracy: 0.6932 - department_output_loss: 1.5979 - loss: 2.9313 


In [21]:
# Print the accuracy for both department and attrition
test_results = model.evaluate(X_test_scaled, [y_test_dept_encoded, y_test_attrition_encoded])

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - attrition_output_accuracy: 0.8136 - attrition_output_loss: 1.3241 - department_output_accuracy: 0.6932 - department_output_loss: 1.5979 - loss: 2.9313


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1. Yes. The accuracy is the best metric because it tells us how much the prediction would be correct and it also shows the loss
2. The activation functions I used the initial layers are relu. This is for the input and hidden layers. For the final output layers, I used softmax as there are multiple outputs (department, attrition ) .
3.The model can be improved by adding more hidden layers which will improve the accuracy output. Also, by running more epochs loss can be reduced a bit more which will improve the overall model.